# 00 · Preparar o Drive — **rodar em casa, uma vez só**

Este notebook não é de palco. Ele cria a estrutura de pastas no seu Drive,
baixa os modelos e deixa tudo pronto para as demos rodarem **sem depender de
download na hora da palestra**.

Raiz de tudo: `/content/drive/MyDrive/PALESTRA-IA`

**Você não precisa juntar foto nenhuma.** Desde 22/08/2026 as demos de treino
(04 e 05) coletam o material **na hora, pela webcam**, com o estúdio de coleta.
Este notebook só prepara o terreno.

O que ele faz:

| passo | o quê |
|---|---|
| 1 | monta a árvore de pastas no seu Drive |
| 2 | baixa os quatro modelos base e guarda lá |
| 3 | confere que tudo está no lugar |

O passo 2 é o que importa: com os modelos já no Drive, **a palestra não depende
do download na hora**, que é onde o Wi-Fi de evento costuma falhar.

O que levar na mala, isso sim: **um capacete e um óculos de proteção** (demo 05)
e **três produtos que se distingam de longe** (demo 04).

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

# ── 2. a raiz de tudo, e a conferência de que ela é REAL ──────────────
#
# ARMADILHA que já custou uma sessão: a linha acima cria a variável
# `drive` (minúscula), que é o MÓDULO do Colab. Se algum caminho for
# escrito com `drive` em vez de `DRIVE`, o Python aceita numa boa e
# monta um caminho como
#     <module 'google.colab.drive' from '/usr/local/...'>/04-garrafas
# O código roda, cria pastas, exporta arquivos — tudo no disco
# temporário do Colab, que evapora quando a sessão encerra. Nada disso
# chega ao seu Drive, e não há erro nenhum na tela.
#
# A conferência abaixo transforma esse silêncio num aviso imediato.

DRIVE = "/content/drive/MyDrive/PALESTRA-IA"

if not DRIVE.startswith("/content/drive/"):
    raise SystemExit(
        "DRIVE aponta para fora do Google Drive: " + repr(DRIVE) + "\n"
        "Provavelmente algum caminho usou `drive` (o módulo) em vez de `DRIVE`.")
if not os.path.isdir("/content/drive/MyDrive"):
    raise SystemExit("O Drive não montou. Rode esta célula de novo e autorize o acesso.")

os.makedirs(DRIVE, exist_ok=True)
print("raiz no Drive:", DRIVE)
print("existe de verdade:", os.path.isdir(DRIVE))

In [ ]:
# ── 2. cria a arvore de pastas ──
import os
PASTAS = [
    "00-pesos",
    "01-deteccao/entrada",
    "01-deteccao/saida",
    "02-segmentacao/entrada",
    "02-segmentacao/saida",
    "03-pose/entrada",
    "03-pose/saida",
    "04-estoque",
    "04-estoque/pesos",
    "05-epi",
    "05-epi/pesos",
    "99-reserva"
]
for p in PASTAS:
    os.makedirs(os.path.join("/content/drive/MyDrive/PALESTRA-IA", p), exist_ok=True)
print(f"{len(PASTAS)} pastas prontas em {DRIVE}")
for p in PASTAS:
    print("  ", p)

In [ ]:
# ── 3. baixa os modelos base e guarda no Drive ──
#    (baixar agora = na palestra nada depende da internet do local)
import shutil, os
DESTINO = "/content/drive/MyDrive/PALESTRA-IA/00-pesos"
MODELOS = ["yolo11n.pt", "yolo11n-seg.pt", "yolo11n-pose.pt", "yolo11n-cls.pt"]

for m in MODELOS:
    if os.path.exists(f"{DESTINO}/{m}"):
        print("ja tenho:", m); continue
    YOLO(m)                       # baixa para o diretorio corrente
    shutil.copy(m, f"{DESTINO}/{m}")
    print("baixado  :", m)

print("\nconteudo de", DESTINO)
for f in sorted(os.listdir(DESTINO)):
    print("  ", f, round(os.path.getsize(f"{DESTINO}/{f}")/1e6, 1), "MB")

In [ ]:
# ── 4. conferencia final: da para subir no palco? ──
import os, glob

pesos = sorted(glob.glob(f"{DRIVE}/00-pesos/*.pt"))
print("modelos base no Drive:", len(pesos))
for f in pesos:
    print("  ", os.path.basename(f), round(os.path.getsize(f)/1e6, 1), "MB")

faltando = [m for m in ["yolo11n.pt", "yolo11n-seg.pt", "yolo11n-pose.pt", "yolo11n-cls.pt"]
            if not os.path.exists(f"{DRIVE}/00-pesos/{m}")]
print()
if faltando:
    print("FALTA baixar:", ", ".join(faltando), "— rode a celula anterior de novo")
else:
    print("os quatro modelos estao no Drive. A palestra nao depende mais de download.")

# material de eventos anteriores, se houver
print()
for demo in ["04-estoque", "05-epi"]:
    bruto = f"{DRIVE}/{demo}/_bruto"
    if os.path.isdir(bruto):
        classes = sorted(os.listdir(bruto))
        total = sum(len(glob.glob(os.path.join(bruto, c, "*.jpg"))) for c in classes)
        print(f"{demo}: {total} imagens ja coletadas em {len(classes)} classe(s) — {', '.join(classes)}")
    else:
        print(f"{demo}: sem material ainda (normal — ele nasce no palco)")

## Como as pastas viram aprendizado

O YOLO de classificação lê a estrutura de pastas **como se fosse o gabarito**:
o nome da pasta é a resposta certa daquelas imagens. Você não precisa marcar
nada, desenhar caixa nem instalar ferramenta de anotação — basta separar.

```
05-epi/
├── _bruto/                 ← o que a camera coletou, sem tratar
│   ├── com_epi/
│   └── sem_epi/
└── treino/                 ← montado por preparar_dataset, descartavel
    ├── train/              ← o modelo estuda por aqui
    │   ├── com_epi/
    │   └── sem_epi/
    └── val/                ← a prova: fotos que ele nunca viu
        ├── com_epi/
        └── sem_epi/
```

É essa a frase para o palco: **"eu não programei nenhuma regra sobre capacete.
Eu só separei as fotos em duas pastas."**

O `_bruto/` **acumula entre eventos**: o material da palestra passada continua
lá, e a coleta da próxima só soma. O `treino/` é remontado do zero a cada
execução, com semente fixa, então ensaio e palco veem a mesma divisão.